# Week 3: Statistics Recap

**Foundations of Data Science**

## In this lesson

1. Describing a distribution: location, spread, shape
2. Is it normal, and does it matter?
3. A statistic is a guess with an error bar: standard errors, confidence intervals, the bootstrap
4. Survey weights: what a row stands for
5. Comparing groups: Table 1, t-tests, chi-square, and what a p-value does and doesn't say
6. Effect size vs significance, and many comparisons at once

## The one idea

**Every number you compute from a sample is an estimate, and an estimate without an error bar is a guess.** Last week we said diabetes prevalence among 30-39 year olds is 3.5%. Today we ask: 3.5% plus or minus what? And when two groups differ, how do we say whether the difference is real, how big it is, and whether it matters to the clinic?

Same decision as before: screen all adults from 35, or only those who are overweight or obese. The clinic's director now wants to know how much to trust the numbers.

## Setup

<!-- nb-links -->
[Download this notebook](week_03_statistics_recap.ipynb?download=1){download="week_03_statistics_recap.ipynb"} &nbsp;|&nbsp; [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshpaner/fds_course_students/blob/main/notebooks/week_03/week_03_statistics_recap.ipynb)


In [ ]:
# --- Setup: works in Colab and locally ---------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "fds_course"

if IN_COLAB:
    # Clone the course repo and move into it (skip if already there)
    if not Path(REPO).exists():
        !git clone -q https://github.com/lshpaner/{REPO}.git
    os.chdir(REPO)
    !pip install -q -r requirements.txt
else:
    # Local: the notebook lives in notebooks/week_XX/, so go up two levels to the repo root
    if Path.cwd().name.startswith("week_"):
        os.chdir(Path.cwd().parents[1])

# Make core/ importable
sys.path.insert(0, os.getcwd())

# Fetch this week's raw data only if it isn't already on disk, then build the processed table
for component in ["nhanes_demo", "nhanes_bmx", "nhanes_bpx", "nhanes_diq"]:
    if not Path(f"data/raw/{component}.parquet").exists():
        !python data_gen.py fetch --name {component}
if not Path("data/processed/nhanes.parquet").exists():
    !python preprocessing.py

print("Working directory:", os.getcwd())

## 1. Describing a distribution

Three things summarize a numeric column: **where it sits** (mean, median), **how spread out it is** (standard deviation, interquartile range), and **what shape it has** (symmetric, skewed, bimodal). Mean and SD are the right pair for symmetric data; median and IQR for skewed data, because a few extreme values drag the mean but not the median.

We'll work with adults only, as established last week.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

nhanes = pd.read_parquet("data/processed/nhanes.parquet")
adults = nhanes[nhanes["age"] >= 20].copy()
adults["sex"] = adults["sex"].map({1: "male", 2: "female"})
adults["age_band"] = pd.cut(adults["age"], bins=[20, 40, 60, 81], labels=["20-39", "40-59", "60+"], right=False).astype(str)

# Location, spread, and a skew measure for the three measurements we care about
desc = adults[["age", "bmi", "sbp"]].describe().T
desc["skew"] = adults[["age", "bmi", "sbp"]].skew()
desc.round(1)

In [ ]:
from eda_toolkit import plot_distributions

# Histogram + KDE with mean and median marked. When mean sits to the right of median, the tail is on the right.
plot_distributions(
    adults, vars_of_interest=["age", "bmi", "sbp"],
    n_rows=1, n_cols=3, subplot_figsize=(15, 4),
    plot_mean=True, plot_median=True, mean_color="red", median_color="black",
)

Age is flat-ish (NHANES oversamples older adults by design, more on that in section 4). BMI and blood pressure are right-skewed: most people cluster in the middle and a tail runs out to the right. For those two, report the median and IQR alongside the mean, and expect the mean to overstate the "typical" person.

## 2. Is it normal, and does it matter?

The normal (bell-shaped) distribution matters because a lot of classical statistics assumes it. Two ways to check:

- **A QQ plot**: sort your data, sort the same number of draws from a perfect normal, plot one against the other. If the data are normal the points fall on a straight line; a right skew bends the upper end upward.
- **A formal test** (D'Agostino, Anderson-Darling): gives a p-value for "is this normal?" With 5,000 rows these tests reject almost anything, because real data are never *exactly* normal. The plot is more useful than the p-value here.

Does it matter? Less than people think. The t-test and the mean's confidence interval hold up well for skewed data when n is in the thousands, because of the central limit theorem (section 3). It matters more when n is small, or when you're modeling the variable itself (week 6).

In [ ]:
from eda_toolkit import normality_tests, distribution_gof_plots

# Formal tests. Note the library skips Shapiro-Wilk above n=5000 and tells you why.
normality_tests(adults, features=["bmi", "sbp"], tests=["dagostino", "anderson"])

In [ ]:
# QQ plot for BMI: the right tail peels away from the line, which is the skew we saw in the histogram
distribution_gof_plots(adults.dropna(subset=["bmi"]), var="bmi", dist="norm", plot_types=["qq"], figsize=(5, 5))

In [ ]:
# A log transform often straightens a right-skewed variable. Compare the QQ plot.
adults["log_bmi"] = np.log(adults["bmi"])
distribution_gof_plots(adults.dropna(subset=["log_bmi"]), var="log_bmi", dist="norm", plot_types=["qq"], figsize=(5, 5))

## 3. A statistic is a guess with an error bar

We computed 15.7% diabetes prevalence among 5,565 adults. If NHANES had sampled a different 5,565 adults, the number would be a bit different. The **standard error** (SE) is the typical size of that "a bit." For a proportion p from n people:

$$\text{SE} = \sqrt{\frac{p(1-p)}{n}}$$

In words: the uncertainty shrinks with the square root of the sample size, so four times the data buys you half the error bar. A **95% confidence interval** is roughly the estimate plus or minus two SEs. It means: if we repeated the survey many times and built this interval each time, about 95% of those intervals would contain the true population value.

Two ways to get the interval. The formula above, or the **bootstrap**: resample your own data with replacement many times, recompute the statistic each time, and look at how much it bounces around. The bootstrap needs no formula and works for any statistic, which is why it's the workhorse in practice.

In [ ]:
# Prevalence in the 30-39 band: formula CI vs bootstrap CI
band = adults[(adults["age"] >= 30) & (adults["age"] < 40)]
p = band["diabetes"].mean()
n = len(band)
se = np.sqrt(p * (1 - p) / n)
print(f"n = {n},  prevalence = {p:.3f},  SE = {se:.4f}")
print(f"formula 95% CI: [{p - 1.96*se:.3f}, {p + 1.96*se:.3f}]")

# Bootstrap: 2,000 resamples of the same size, with replacement
rng = np.random.default_rng(0)
boot = np.array([rng.choice(band["diabetes"].values, size=n, replace=True).mean() for _ in range(2000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"bootstrap 95% CI: [{lo:.3f}, {hi:.3f}]")

In [ ]:
import matplotlib.pyplot as plt

# The bootstrap distribution is the error bar, drawn out. Its spread is the SE; its middle 95% is the CI.
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(boot, bins=40, color="C0", alpha=0.8)
ax.axvline(p, color="black", label="observed")
ax.axvline(lo, color="red", linestyle="--", label="95% CI")
ax.axvline(hi, color="red", linestyle="--")
ax.set_xlabel("Bootstrap prevalence, age 30-39")
ax.set_title("2,000 resamples: the estimate is 3.5% but could plausibly be 2.3% to 4.9%")
ax.legend()
plt.show()

Read that back to the clinic: prevalence in 30-39 year olds is about 3.5%, and the data are consistent with anything from roughly 2.5% to 5%. That range is what the director should plan around, not the point estimate.

The bootstrap also previews week 8: bootstrapped confidence intervals around model metrics (AUC, calibration) are how we'll report model performance honestly.

## 4. Survey weights: what a row stands for

NHANES is not a simple random sample. The CDC deliberately **oversamples** some groups (older adults, some racial and ethnic groups, lower-income households) to get enough of them to say something precise, then provides a **weight** per person: the number of US residents that row represents. Each `weight_mec` here is in the tens of thousands.

Consequences:
- An **unweighted** mean describes *the sample*, which is a distorted mirror of the country.
- A **weighted** mean describes *the population*: multiply each person's value by their weight, sum, divide by the sum of weights.
- The weight to use depends on which component defined your cohort. Our diabetes question comes from the interview, but BMI and blood pressure come from the exam, so we use the exam weight (`WTMEC2YR`) and restrict to people who were examined (weight > 0).
- Correct *standard errors* under a survey design also need the strata and cluster variables (`SDMVSTRA`, `SDMVPSU`), which is beyond today. The point for now is to know that the naive SE is a little too optimistic for survey data, and that packages exist for this (`statsmodels` survey tools, `samplics`).

In [ ]:
# Weighted vs unweighted prevalence, overall and by age band, among examined adults
examined = adults[adults["weight_mec"] > 0].copy()

def weighted_mean(df: pd.DataFrame, col: str, w: str = "weight_mec") -> float:
    return np.average(df[col], weights=df[w])

overall = pd.Series({
    "unweighted": examined["diabetes"].mean(),
    "weighted": weighted_mean(examined, "diabetes"),
}).round(3)
print("overall adult prevalence:\n", overall, "\n")

by_band = examined.groupby("age_band").apply(
    lambda g: pd.Series({"unweighted": g["diabetes"].mean(), "weighted": weighted_mean(g, "diabetes"), "n": len(g)})
).round(3)
by_band

The weighted prevalence is lower than the unweighted one: 11.5% against 15.7% overall. Part of that is age, since the sample over-represents older adults. But the gap persists *within* age bands (23.6% vs 28.5% at 60+), so age isn't the whole story: NHANES also oversamples Hispanic, Black, and lower-income adults, groups with higher diabetes prevalence. The weighted number is the one to quote as a national figure; the unweighted one describes the people who happened to be in the sample.

## 5. Comparing groups

The clinic's real question is a comparison: do people with diabetes differ from people without, and on what? In clinical research the standard first answer is a **Table 1**: every variable summarized overall and by group, with a test of whether the groups differ. `generate_table1` builds it, and chooses the test for you:

- **Welch's t-test** for a continuous variable: are the two group means different? Welch's version doesn't assume equal variances, which is the safe default.
- **Chi-square test** for a categorical variable: is the distribution across categories different between groups?

A **p-value** is the probability of seeing a difference at least this large *if the groups were truly identical*. Small p (conventionally under 0.05) means "this would be surprising under no difference." It does **not** mean the difference is large, important, or causal, and with 5,000 people almost everything comes out significant. Section 6 deals with that.

In [ ]:
from eda_toolkit import generate_table1

table1 = generate_table1(
    adults[["diabetes", "sex", "age_band", "age", "bmi", "waist_cm", "sbp", "dbp", "income_ratio"]],
    groupby_col="diabetes",
    categorical_cols=["sex", "age_band"],
    continuous_cols=["age", "bmi", "waist_cm", "sbp", "dbp", "income_ratio"],
    value_counts=True,
    decimal_places=1,
)
table1

In [ ]:
from eda_toolkit import grouped_distributions

# Distributions by group, overlaid. A difference in means can hide very different shapes.
grouped_distributions(
    adults.dropna(subset=["bmi", "sbp"]), features=["bmi", "sbp"], by="diabetes",
    n_rows=1, n_cols=2, figsize=(12, 4), bins=40,
)

Read the table like a clinician: people with diabetes are older (64 vs 49), heavier (BMI 32 vs 29, waist 109 vs 99 cm), have higher systolic pressure, and are more often male. Every p-value rounds to zero. Income ratio is the exception: 2.5 vs 2.6, p around 0.1, so no evidence of a difference there.

The overlaid distributions add something the table can't: the BMI curves overlap heavily. The *average* diabetic is heavier, but plenty of diabetics have a normal BMI and plenty of non-diabetics are obese. That overlap is what will make classification hard in week 7, and it's why a screening rule based on weight alone misses people.

## 6. Effect size vs significance, and many comparisons

Two corrections to how people read a Table 1.

**Significance isn't size.** With n = 5,565, a BMI difference of 0.3 would be "significant." What the clinic needs is the *size* of the difference and its interval. For a difference in means, the confidence interval does both jobs: it tells you how big, and whether zero is plausible. A standardized version, **Cohen's d** (difference divided by pooled SD), lets you compare effects across variables with different units; rough guide, 0.2 small, 0.5 medium, 0.8 large.

**Many tests, many false alarms.** Run 20 tests at the 0.05 level on groups that truly don't differ and you expect one false positive. Table 1 ran 8. Two standard fixes: **Bonferroni** (divide 0.05 by the number of tests; simple, conservative) and **Benjamini-Hochberg** (controls the share of false discoveries; less conservative, the usual choice). `generate_table1` can apply either.

In [ ]:
from scipy import stats

# Effect size with a confidence interval for the BMI difference, computed by hand so the pieces are visible
g1 = adults.loc[adults["diabetes"] == 1, "bmi"].dropna()
g0 = adults.loc[adults["diabetes"] == 0, "bmi"].dropna()

diff = g1.mean() - g0.mean()
se_diff = np.sqrt(g1.var(ddof=1) / len(g1) + g0.var(ddof=1) / len(g0))   # Welch SE
pooled_sd = np.sqrt((g1.var(ddof=1) + g0.var(ddof=1)) / 2)
d = diff / pooled_sd

print(f"BMI difference: {diff:.2f}  95% CI [{diff - 1.96*se_diff:.2f}, {diff + 1.96*se_diff:.2f}]")
print(f"Cohen's d: {d:.2f}  (about 0.4: a small-to-medium effect, despite p < 0.001)")

# And the same from the library's wrapper, which returns the t-test in one call
stats.ttest_ind(g1, g0, equal_var=False)

In [ ]:
# Table 1 again with Benjamini-Hochberg adjusted p-values. With differences this large nothing changes;
# the habit matters for the project, where you'll test more things on less data.
generate_table1(
    adults[["diabetes", "sex", "age_band", "age", "bmi", "waist_cm", "sbp", "dbp", "income_ratio"]],
    groupby_col="diabetes",
    categorical_cols=["sex", "age_band"],
    continuous_cols=["age", "bmi", "waist_cm", "sbp", "dbp", "income_ratio"],
    decimal_places=1,
    apply_bh_fdr=True,
)

## What to carry forward

- Report medians and IQRs for skewed variables; check shape with a histogram and a QQ plot, not a p-value.
- Every estimate gets an interval. The bootstrap gives you one for anything.
- Survey data: use the weights for population numbers, and know that naive SEs are a bit too small.
- Table 1 is the first thing a clinical reader looks for. Build it, then read the effect sizes, not the stars.
- Many tests need a correction; BH is the default.

## Looking ahead
- **Project**: the one-page problem statement is due in week 4. Include at least one number with an interval around it.
- **Reading**: Bruce & Bruce chapters 2 and 3 (sampling distributions, statistical experiments). ISLR 2.2 (assessing model accuracy) to set up weeks 6 through 8.